# Experimento 3 — Versión PRUEBA (_sergi)

Notebook standalone con la **nueva consulta directa a BigQuery raw** y features ajustadas.
Todos los artefactos se suben con sufijo `_sergi` → no afecta producción.

**Columnas nuevas respecto al Exp 3 original:**
- `neighbourhood_cleansed`, `room_type` (categóricas)
- `listing_price`, `minimum_nights`, `review_scores_rating`
- `has_reviews`, `instant_bookable`, `is_holiday`, `days_to_next_holiday`
- `precipitation_mm`, `num_sports`, `num_festivals`, `total_attendance`

**Eliminadas:** `bedrooms`, `beds` (no están en la nueva query)

## 0. Dependencias

In [ ]:
import sys
!{sys.executable} -m pip install google-cloud-bigquery db-dtypes google-auth xgboost --quiet

## 1. Conexión a BigQuery y carga de datos

In [ ]:
from google.cloud import bigquery
import pandas as pd
import numpy as np

PROJECT_ID = "project3grupo1"
client = bigquery.Client(project=PROJECT_ID)
print("✅ Conectado a BigQuery")

In [ ]:
# ── Nueva consulta (directa a raw tables, sin depender del mart dbt) ─────────
query = """
WITH listings_dedup AS (
    SELECT *
    FROM `project3grupo1.airbnb_raw.listings`
    QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY snapshot_date DESC) = 1
),

events_agg AS (
    SELECT
        start_date,
        COUNTIF(category = 'sports')   as num_sports,
        COUNTIF(category = 'festivals') as num_festivals,
        SUM(phq_attendance)             as total_attendance
    FROM `project3grupo1.airbnb_features.events`
    GROUP BY start_date
),

holidays_valencia AS (
    SELECT DISTINCT date
    FROM `project3grupo1.airbnb_features.holidays`
    WHERE applies_to_valencia = True
),

calendar_dates AS (
    SELECT DISTINCT date
    FROM `project3grupo1.airbnb_raw.calendar`
),

days_to_holiday AS (
    SELECT
        cd.date,
        MIN(DATE_DIFF(h.date, cd.date, DAY)) as days_to_next_holiday
    FROM calendar_dates cd
    LEFT JOIN holidays_valencia h ON h.date >= cd.date
    GROUP BY cd.date
)

SELECT
    -- Target
    CASE WHEN c.available = False THEN 1 ELSE 0 END as is_occupied,

    -- Temporales
    c.date,
    EXTRACT(MONTH      FROM c.date) as month,
    EXTRACT(DAYOFWEEK  FROM c.date) as day_of_week,
    IF(EXTRACT(DAYOFWEEK FROM c.date) IN (1,7), 1, 0) as is_weekend,
    COALESCE(dh.days_to_next_holiday, 0)  as days_to_next_holiday,
    IF(h.date IS NOT NULL, 1, 0)          as is_holiday,

    -- Del piso
    l.neighbourhood_cleansed,
    l.room_type,
    l.accommodates,
    l.price                               as listing_price,
    l.minimum_nights,
    l.number_of_reviews,
    l.review_scores_rating,
    IF(l.number_of_reviews > 0, 1, 0)     as has_reviews,
    l.instant_bookable,

    -- Meteorología
    w.temp_mean,
    w.precipitation_mm,

    -- Eventos
    COALESCE(e.num_sports,       0) as num_sports,
    COALESCE(e.num_festivals,    0) as num_festivals,
    COALESCE(e.total_attendance, 0) as total_attendance

FROM `project3grupo1.airbnb_raw.calendar` c
LEFT JOIN listings_dedup                   l  ON c.listing_id = l.id
LEFT JOIN `project3grupo1.airbnb_features.weather` w ON c.date = w.date
LEFT JOIN events_agg                       e  ON c.date = e.start_date
LEFT JOIN holidays_valencia                h  ON c.date = h.date
LEFT JOIN days_to_holiday                  dh ON c.date = dh.date
WHERE c.date <= '2026-05-19'
QUALIFY ROW_NUMBER() OVER (PARTITION BY c.listing_id, c.date ORDER BY c.date DESC) = 1
"""

df = client.query(query).to_dataframe()
print(f"✅ Filas cargadas: {len(df):,}")
print(f"   Columnas: {list(df.columns)}")
df.head()

## 2. EDA básico

In [ ]:
print("Shape:", df.shape)
print("\nDistribución is_occupied:")
print(df["is_occupied"].value_counts(normalize=True).round(3))
print("\nNulos por columna:")
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
print(df.dtypes)

## 3. Preparación de features

Features ajustadas a la nueva consulta.
- `neighbourhood_cleansed` y `room_type` → categóricas (XGBoost con `enable_categorical=True`)
- `instant_bookable` → cast a int
- Nulos numéricos → 0
- `bedrooms` y `beds` ya no están en la consulta → eliminadas

In [ ]:
# ── Features numéricas ────────────────────────────────────────────────────────
NUM_FEATURES = [
    # Temporales
    "month",
    "day_of_week",
    "is_weekend",
    "days_to_next_holiday",
    "is_holiday",
    # Listing
    "accommodates",
    "listing_price",
    "minimum_nights",
    "number_of_reviews",
    "review_scores_rating",
    "has_reviews",
    "instant_bookable",
    # Meteorología
    "temp_mean",
    "precipitation_mm",
    # Eventos
    "num_sports",
    "num_festivals",
    "total_attendance",
]

# ── Features categóricas ──────────────────────────────────────────────────────
CAT_FEATURES = [
    "neighbourhood_cleansed",
    "room_type",
]

ALL_FEATURES = NUM_FEATURES + CAT_FEATURES

# ── Procesamiento ─────────────────────────────────────────────────────────────
# instant_bookable puede venir como bool de BigQuery
df["instant_bookable"] = df["instant_bookable"].astype(int)

# Categóricas → dtype category (necesario para XGBoost con enable_categorical)
for col in CAT_FEATURES:
    df[col] = df[col].astype("category")

# Numéricas → rellenar nulos con 0
df[NUM_FEATURES] = df[NUM_FEATURES].fillna(0)

X = df[ALL_FEATURES].copy()
y = df["is_occupied"].astype(int)

print(f"X shape: {X.shape}")
print(f"y distribution:\n{y.value_counts(normalize=True).round(3)}")
print(f"\nFeatures numéricas ({len(NUM_FEATURES)}): {NUM_FEATURES}")
print(f"Features categóricas ({len(CAT_FEATURES)}): {CAT_FEATURES}")

## 4. Entrenamiento XGBoost

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report
from xgboost import XGBClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Peso para compensar el desbalance entre clase 0 y clase 1
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f"Scale pos weight: {scale_pos_weight:.4f}")

# Modelo XGBoost (mismo que Exp 3, + enable_categorical para neighbourhood/room_type)
model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=4,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    enable_categorical=True,   # ← necesario para neighbourhood_cleansed y room_type
    tree_method="hist",        # ← requerido con enable_categorical
)

model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_test  = model.predict(X_test)

train_acc    = accuracy_score(y_train, y_pred_train)
test_acc     = accuracy_score(y_test, y_pred_test)
balanced_acc = balanced_accuracy_score(y_test, y_pred_test)

print(f"\n✅ Entrenamiento completado")
print(f"Train accuracy : {train_acc:.4f}")
print(f"Test  accuracy : {test_acc:.4f}")
print(f"Balanced acc   : {balanced_acc:.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred_test))

In [ ]:
# Búsqueda de threshold óptimo
y_prob_test = model.predict_proba(X_test)[:, 1]

best_threshold = None
best_bal_acc   = 0
best_y_pred    = None

print("📊 Evaluación por threshold:")
for threshold in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    y_pred_thr  = (y_prob_test >= threshold).astype(int)
    bal_acc_thr = balanced_accuracy_score(y_test, y_pred_thr)
    acc_thr     = accuracy_score(y_test, y_pred_thr)
    print(f"  threshold={threshold}  acc={acc_thr:.4f}  balanced_acc={bal_acc_thr:.4f}")
    if bal_acc_thr > best_bal_acc:
        best_bal_acc   = bal_acc_thr
        best_threshold = threshold
        best_y_pred    = y_pred_thr

print(f"\n🏆 Mejor threshold: {best_threshold}  (balanced_acc={best_bal_acc:.4f})")
print(classification_report(y_test, best_y_pred))

In [ ]:
import matplotlib.pyplot as plt

importances = model.feature_importances_
feat_imp = pd.Series(importances, index=ALL_FEATURES).sort_values(ascending=True)

feat_imp.plot(kind="barh", figsize=(8, 6), title="Feature Importance (gain)")
plt.tight_layout()
plt.show()

---
## 5. Subida de artefactos a GCS (sufijo `_sergi`)

Destino: `gs://project3grupo1-ml-models/occupancy_sergi/`

In [ ]:
import os, importlib.util

HERE = os.path.abspath("")
if HERE not in __import__("sys").path:
    __import__("sys").path.insert(0, HERE)

import config_sergi as CFG

print(f"✅ Config cargada:")
print(f"   GCS_BUCKET   : {CFG.GCS_BUCKET}")
print(f"   MODEL_PREFIX : {CFG.MODEL_PREFIX}")
print(f"   MODEL_NAME   : {CFG.MODEL_DISPLAY_NAME}")
print(f"   ENDPOINT     : {CFG.ENDPOINT_DISPLAY_NAME}")

In [ ]:
_spec = importlib.util.spec_from_file_location(
    "upload_artifacts",
    os.path.join(HERE, "02_upload_artifacts.py")
)
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
ArtifactUploader = _mod.ArtifactUploader
print("✅ ArtifactUploader cargado")

In [ ]:
uploader = ArtifactUploader(CFG)

# 1. JSONs de configuración (bq_config + feature_config)
uploader.upload_configs()

# 2. Modelo entrenado
uploader.upload_pkl(model, CFG.ARTIFACTS["model"])
print(f"✅ Modelo → gs://{CFG.GCS_BUCKET}/{CFG.ARTIFACTS['model']}")

# 3. Verificación
uploader.verify()
uploader.list_blobs()

---
## 6. Build Docker + Registro en Vertex AI (sufijo `_sergi`)

> ⚠️ Requiere `gcloud` autenticado. Tarda ~3-5 min.

In [ ]:
_reg_spec = importlib.util.spec_from_file_location(
    "registry_endpoint",
    os.path.join(HERE, "03_registry_endpoint.py")
)
_reg_mod = importlib.util.module_from_spec(_reg_spec)
_reg_spec.loader.exec_module(_reg_mod)
VertexDeployer = _reg_mod.VertexDeployer
print("✅ VertexDeployer cargado")

In [ ]:
deployer = VertexDeployer(CFG)
deployer.build_and_push()
print(f"✅ Imagen disponible: {CFG.IMAGE}")

In [ ]:
model_vertex = deployer.register_model()
print(f"✅ Modelo registrado: {model_vertex.resource_name}")

---
## 7. (Opcional) Deploy del endpoint `occupancy-predictor-endpoint-sergi`

> ⚠️ Tarda ~10-20 min.

In [ ]:
endpoint = deployer.create_endpoint()
deployer.deploy(model_vertex, endpoint)
print(f"✅ Endpoint listo: {endpoint.resource_name}")